# 03 — Classifier Baseline (Policy Engine)

Train SVM and Random Forest Policy Engines on **human data only** and report TAR / FAR / EER.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_dataset, list_subjects
from src.data.splits import build_subject_split, build_training_dataset, build_verification_dataset
from src.models.classifier import train_policy_engine, predict_proba
from src.evaluation.metrics import compute_metrics

df = load_dataset()
subjects = list_subjects(df)[:5]  # quick demo; scale to all for final results
rows = []
for subject in subjects:
    split = build_subject_split(df, subject, 'hold_flight')
    Xtr, ytr = build_training_dataset(split, impostor_samples_per_user=640)
    Xv, yv = build_verification_dataset(split)
    for clf in ('svm', 'random_forest'):
        eng = train_policy_engine(
            Xtr, ytr, subject=subject, feature_set='hold_flight',
            classifier_type=clf, tune_hyperparameters=False,
        )
        scores = predict_proba(eng, Xv)
        m = compute_metrics(yv, scores, subject=subject)
        rows.append({**m.__dict__, 'classifier': clf})
        print(subject, clf, f"TAR={m.tar:.3f} FAR={m.far:.3f} EER={m.eer:.3f}")

baseline = pd.DataFrame(rows)
baseline.groupby('classifier')[['tar','far','eer']].mean()